<table align="left"><tr><td>
<a href="https://colab.research.google.com/github/kikim6114/nlp2026/blob/main/05.Language_Model-1.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="코랩에서 실행하기"/></a>
</td></tr></table>

In [ ]:
# Colab이나 Kaggle을 사용하는 경우가 아니면, 이 cell을 모두 주석화하여 skip할 것.
import os

repository_name = 'nlp2026'
repository_url = f'https://github.com/kikim6114/{repository_name}.git'

# 항상 루트 경로(/content/)로 이동 후 확인
%cd /content/

if not os.path.exists(repository_name):
    !git clone {repository_url}
    print(f"{repository_name} 클론 완료")
else:
    print(f"{repository_name} 폴더가 이미 존재합니다. 클론을 건너뜁니다.")
%cd nlp2026

<font face="Times New Roman" size=7 color='blue'>5. 언어 모델(Language Model)-1<font>

[알림] 이 자료는 Andrej Karpathy의 makemore 강의 자료를 기반으로 수정된 자료입니다.<br>
([[youtube link](https://www.youtube.com/watch?v=PaCmpygFfXo)][[notebook]((https://github.com/karpathy/nn-zero-to-hero/blob/master/lectures/makemore/makemore_part1_bigrams.ipynb))])

## Bigram 언어 모델
$$p(w_1, w_2, \ldots, w_T) = \prod_{t=1}^{T} p(w_t \mid w_{t-1})$$
- $p(w_t \mid w_{t-1})$: 이전 토큰 $w_{t-1}$이 주어질 때 $w_t$의 조건부 확률.
- `<s>`: 시작 토큰, 즉 $w_0$
- `</s>`: 종료 토큰, 즉 $w_T$

In [ ]:
data = open('names.txt').read().splitlines()
data[:10]

- 데이터셋은 사람 이름들로 구성되어 있다.
- 간단하게 처리하기 위해 **문자(character) 단위 토크나이제이션**을 사용한다. 즉, 어휘(vocabulary)는 영어 소문자 26개 + `<s>` (시작 토큰) + `</s>` (종료 토큰) 총 28개이다.

In [ ]:
['<s>'] + list(data[0]) + ['</s>']

### Bigram 모델 학습

- 데이터에서 각 토큰 $w_t$가 $w_{t-1}$ 다음에 등장하는 빈도를 기반으로 다음 토큰 확률을 설정하여 bigram 모델을 "학습"한다:

\begin{align*}
p(w_t \mid w_{t-1}) 
= \frac{\text{count}(w_{t-1}, w_t)}{\sum_{w'} \text{count}(w_{t-1}, w')}.
\end{align*}

- 이 방식은 전체 코퍼스의 **최대가능도(maximum likelihood)** 를 구하는 것과 동일하다. 즉, 다음을 최대화한다:
\begin{align*}
\prod_{\substack{\text{all pairs }(w_{t-1}, w_t) \\ \text{in the corpus}}}
p(w_t \mid w_{t-1}),
\end{align*}

- 표준적인 수학적 논증을 통해 위의 빈도 기반 공식이 이를 최대화함을 증명할 수 있다.

In [ ]:
# Step 1: 바이그램 등장 횟수를 센다
bigram_counts = {}
for w in data:
    sequence = ['<s>'] + list(w) + ['</s>']
    for w1, w2 in zip(sequence, sequence[1:]):
        bigram = (w1, w2)
        bigram_counts[bigram] = bigram_counts.get(bigram, 0) + 1

# 문자 ↔ 인덱스 간의 변환 딕셔너리를 만듭니다
char_to_index = {char: i for i, char in enumerate('abcdefghijklmnopqrstuvwxyz')}
char_to_index['<s>'] = 26
char_to_index['</s>'] = 27
index_to_char = {i: char for char, i in char_to_index.items()}

sorted(bigram_counts.items(), key=lambda w: w[1], reverse=True)[:10]

In [ ]:
# Step 2: 전이 확률(transition probabilities) 계산
import torch
P = torch.zeros((28, 28), dtype=torch.float32)

for (w1, w2), count in bigram_counts.items():
    i = char_to_index[w1]
    j = char_to_index[w2]
    P[i, j] = count

# 각 행을 정규화(행의 합이 1이 되도록)
P /= P.sum(dim=1, keepdim=True)

### 확률 행렬을 시각화

In [ ]:
import matplotlib.pyplot as plt
%matplotlib inline

plt.figure(figsize=(25,25))
plt.imshow(P, cmap='Blues')
for i in range(28):
    for j in range(28):
        chstr = index_to_char[i] + index_to_char[j]
        plt.text(j, i, chstr, ha="center", va="bottom", color='gray')
        plt.text(j, i, '%.2E' % P[i, j].item(), ha="center", va="top", color='gray')
plt.axis('off')
plt.savefig('bigram_table.png', dpi=300)

#### **질문**

- 시퀀스의 시작에 나타날 가능성이 가장 높은 토큰은 무엇인가?
- 시퀀스의 끝에 나타날 가능성이 가장 높은 토큰은 무엇인가?
- 토큰 `v` 다음에 나타날 가능성이 가장 높은 토큰은 무엇인가?
- 이 모델에서 확률이 가장 높은 토큰 쌍은 무엇인가?

### Bigram 모델로 시퀀스 생성

학습된 bigram 모델을 사용해 시퀀스를 생성한다. 모델을 $p_\theta$로 나타내며, $\theta$는 모델의 파라미터(즉, 전이 확률)이다.

먼저, 첫 번째 토큰을 샘플링한다:

\begin{align*}
    w_1\sim p_\theta(\cdot|w_0)
\end{align*}

In [ ]:
index = torch.multinomial(      # 입력되는 벡터 또는 텐서(여기서는 P의 한 행)에서 샘플링을 수행하는 함수
    P[char_to_index['<s>']],    # '<s>'에서 시작하는 다음 문자를 샘플링
    num_samples=10,             # 10개의 샘플을 생성
    replacement=True            # 복원 추출
)

[index_to_char[i.item()] for i in index]

전체 시퀀스를 생성하기 위해 `<s>` 토큰에서 시작하여 현재 토큰의 분포를 기반으로 다음 토큰을 반복적으로 샘플링한다. `</s>` 토큰이 샘플링될 때까지 이 과정을 반복하며, 이는 시퀀스의 끝을 의미한다.

단계별 과정:

1. `<s>` 토큰에서 시작
2. 현재 토큰의 확률 분포에서 다음 토큰을 샘플링
3. 샘플링된 토큰을 시퀀스에 추가
4. 샘플링된 토큰을 새로운 현재 토큰으로 사용
5. 다시 `</s>` 토큰이 샘플링될 때까지 2~4단계를 반복

다음 코드 셀에서 구현한다.

In [ ]:
def generate_sequence():
    sequence = ['<s>']
    while True:
        current_char = sequence[-1]
        current_index = char_to_index[current_char]
        next_index = torch.multinomial(P[current_index], num_samples=1).item()
        next_char = index_to_char[next_index]
        if next_char == '</s>':
            break
        sequence.append(next_char)
    return ''.join(sequence[1:])

# 10개의 시퀀스(이름)를 생성
generated_sequences = [generate_sequence() for _ in range(10)]
generated_sequences

모델이 새로운 시퀀스(이름)를 생성하는 데 완벽하지는 않지만, 비교를 위해 모든 문자를 균등 확률(uniform)로 샘플링하는 베이스라인과 비교해보자.

In [ ]:
import random

def generate_uniform_sequence():
    characters = list(char_to_index.keys())
    sequence = ['<s>']
    while True:
        next_char = random.choice(characters)
        if next_char == '</s>':
            break
        sequence.append(next_char)
    return ''.join(sequence[1:])

# 10개의 시퀀스를 생성
uniform_sequences = [generate_uniform_sequence() for _ in range(10)]
uniform_sequences

Bigram 모델이 균등 샘플링 베이스라인보다 훨씬 더 그럴듯한 구조를 갖고 있음을 알 수 있다.

### 로그 가능도(Log-likelihood)와 퍼플렉시티(Perplexity)를 이용한 모델 평가
모델을 평가하기 위해 데이터셋에 있는 시퀀스의 **로그 가능도**를 계산할 수 있다. 시퀀스의 로그 가능도는 다음과 같이 정의된다:

\begin{align*}
\log p(w_1, w_2, \ldots, w_T) = \sum_{t=1}^{T} \log p(w_t \mid w_{t-1})
\end{align*}

여기서 $ p(w_t \mid w_{t-1}) $는 이전 토큰 $ w_{t-1} $이 주어질 때 토큰 $ w_t $의 조건부 확률이다. 많은 확률을 곱할 때 발생하는 **수치 언더플로(numerical underflow)** 를 방지하기 위해 로그 확률을 사용한다.

In [ ]:
def log_likelihood(P, dataset):
    n = 0
    ll = 0
    for w in dataset:
        sequence = ['<s>'] + list(w) + ['</s>']
        for w1, w2 in zip(sequence, sequence[1:]):
            i = char_to_index[w1]
            j = char_to_index[w2]
            ll += torch.log(P[i, j])
            n += 1
    return ll, n

ll, n = log_likelihood(P, data)
print(f'Log likelihood: {ll.item():.4f}')
print(f'Average next-token log likelihood {ll.item() / n:.4f}')

로그 가능도는 **퍼플렉시티(Perplexity)** 로 표현하는 것이 일반적이며, 토큰 단위로 다음과 같이 정의된다:

\begin{align*}
\text{Perplexity} = \exp\left(-\frac{1}{N} \sum_{i=1}^{N} \log p(w_i \mid w_{i-1})\right)
\end{align*}

여기서 $N$은 데이터셋의 전체 토큰 수. **퍼플렉시티가 낮을수록 더 좋은 모델**이다.

퍼플렉시티와 언어 모델 평가 지표에 대한 자세한 내용은 이 [블로그 포스트](https://thegradient.pub/understanding-evaluation-metrics-for-language-models/)를 참고하라.

In [ ]:
def perplexity(model, dataset):
    ll, n = log_likelihood(model, dataset)
    return torch.exp(-ll / n).item()

perplexity(P, data)

#### 홀드아웃 데이터(Hold-out Data)로 평가 

**연습 문제 1**

위에서는 학습 데이터로 모델을 평가했다. 일반화 성능을 평가하려면 별도로 보관해 둔 시퀀스로 평가해야 한다.

이 간단한 예제에서는 테스트용 이름을 따로 분리하지 않았다. 연습으로, 데이터셋을 학습(train), 검증(validation), 테스트(test) 시퀀스로 나누고, 홀드아웃 검증/테스트 퍼플렉시티를 평가해보자.

**[주목]** 학습 중에 보지 못한 시퀀스에 대해 모델이 **확률 0**을 부여할 수 있어 로그 가능도가 $-\infty$가 된다. 이를 해결하기 위해 **균등 스무딩(uniform smoothing)** 을 적용할 수 있다. 즉, 각 다음 토큰 확률에 고정된 확률 값 $\alpha$를 더하는 방법이다.

**연습 문제 2:** 위의 모델을 $n$-gram 모델(예: n = 2, 3, 4, ...)로 일반화하시오.

## 간단한 신경망 Bigram 모델

이제 위의 모델을 **학습** 관점에서 재구성한다. 즉, $p(\cdot|w_{t-1})=W w_{t-1}$ 형태로 표현한다.


In [ ]:
# 학습 데이터셋을 만들기
xs, ys = [], []
for w in data:
    sequence = ['<s>'] + list(w) + ['</s>']
    for w1, w2 in zip(sequence, sequence[1:]):
        xs.append(char_to_index[w1])
        ys.append(char_to_index[w2])

xs = torch.tensor(xs)
ys = torch.tensor(ys)

In [ ]:
xs[:10], ys[:10]

In [ ]:
torch.nn.functional.one_hot(xs, num_classes=28)[:3]

In [ ]:
# W_ij = 문자 i 다음에 문자 j가 올 확률
W = torch.randn(28, 28, requires_grad=True)
learning_rate = 20

def forward(x):
    xenc = torch.nn.functional.one_hot(x, num_classes=28).float()
    logits = xenc.matmul(W) # 차원: (batch_size, 28)
    return logits

def nll_loss(logits, y):
    loss = -torch.nn.functional.log_softmax(logits, dim=1) # 차원: (batch_size, 28)
    loss = loss[range(len(y)), y] # 차원: (batch_size,)
    loss = loss.mean()
    return loss

for k in range(1000):
    logits = forward(xs)
    nll = nll_loss(logits, ys)
    loss = nll 

    W.grad = None
    loss.backward()
    if k % 100 == 0:
        print(loss.item())

    W.data -= learning_rate * W.grad


In [ ]:
# 신경망 모델의 확률 행렬을 시각화
P_ = torch.nn.functional.softmax(W, dim=1).detach()

ll, n = log_likelihood(P_, data)
print(f'Log likelihood: {ll.item():.4f}')
print(f'Average next-token log likelihood {ll.item() / n:.4f}')

plt.figure(figsize=(25,25))
plt.imshow(P_.numpy(), cmap='Blues')
for i in range(28):
    for j in range(28):
        chstr = index_to_char[i] + index_to_char[j]
        plt.text(j, i, chstr, ha="center", va="bottom", color='gray')
        plt.text(j, i, '%.2E' % P_[i, j].item(), ha="center", va="top", color='gray')

#### **질문**

- 시퀀스의 시작에 나타날 가능성이 가장 높은 토큰은 무엇인가?
- 시퀀스의 끝에 나타날 가능성이 가장 높은 토큰은 무엇인가?
- 토큰 `v` 다음에 나타날 가능성이 가장 높은 토큰은 무엇인가?
- 이 모델에서 확률이 가장 높은 토큰 쌍은 무엇인가?

- 간단한 신경망 학습 모델이 빈도 기반 모델과 매우 유사한 결과를 낸다는 것을 확인할 수 있다.

- 다음 노트북에서는 시퀀스를 더 잘 모델링하고 생성하기 위한 더 나은 신경망을 만들어볼 것이다.